In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Określanie obserwabli w bazie Pauliego

<!-- source hash: 014517b1 -->



<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie został opracowany przy użyciu poniższych wymagań.
Zalecamy używanie tych wersji lub nowszych.

```
qiskit[all]~=2.3.0
```
</details>
W mechanice kwantowej obserwable odpowiadają właściwościom fizycznym, które można mierzyć.
Rozważając na przykład układ spinów, możesz być zainteresowany pomiarem energii systemu lub uzyskaniem informacji o wyrównaniu spinów, takich jak namagnesowanie lub korelacje między spinami.

Aby zmierzyć $n$-qubitową obserwablę $O$ na komputerze kwantowym, musisz przedstawić ją jako sumę iloczynów tensorowych operatorów Pauliego, to znaczy

$$
O = \sum_{k=1}^K \alpha_k P_k,~~ P_k \in {I, X, Y, Z}^{\otimes n},~~ \alpha_k \in \mathbb{R},
$$

gdzie

$$
I = \begin{pmatrix}
1 & 0 \\ 0 & 1
\end{pmatrix}
~~
X = \begin{pmatrix}
0 & 1 \\ 1 & 0
\end{pmatrix}
~~
Y = \begin{pmatrix}
0 & -i \\ i & 0
\end{pmatrix}
~~
Z = \begin{pmatrix}
1 & 0 \\ 0 & -1
\end{pmatrix}
$$

i korzystasz z faktu, że obserwabla jest hermitowska, tzn. $O^\dagger = O$. Jeśli $O$ nie jest hermitowska, nadal można ją rozłożyć jako sumę Paulich, ale współczynnik $\alpha_k$ staje się zespolony.

W wielu przypadkach obserwabla jest naturalnie określona w tej reprezentacji po odwzorowaniu badanego układu na Qubity.
Na przykład układ spinów 1/2 można odwzorować na hamiltoniana Isinga

$$
H = \sum_{\langle i, j\rangle} Z_i Z_j - \sum_{i=1}^n X_i,
$$

gdzie indeksy $\langle i, j\rangle$ biegną po oddziałujących spinach, a spiny podlegają poprzecznemu polu w $X$.
Indeks dolny wskazuje, na który Qubit działa operator Pauliego, tzn. $X_i$ stosuje operator $X$ na Qubicie $i$, pozostawiając resztę bez zmian.

W Qiskit SDK ten hamiltoniana można skonstruować następującym kodem.

In [1]:
from qiskit.quantum_info import SparsePauliOp

# define the number of qubits
n = 12

# define the single Pauli terms as ("Paulis", [indices], coefficient)
interactions = [
    ("ZZ", [i, i + 1], 1) for i in range(n - 1)
]  # we assume spins on a 1D line
field = [("X", [i], -1) for i in range(n)]

# build the operator
hamiltonian = SparsePauliOp.from_sparse_list(
    interactions + field, num_qubits=n
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIIIIZZ', 'IIIIIIIIIZZI', 'IIIIIIIIZZII', 'IIIIIIIZZIII', 'IIIIIIZZIIII', 'IIIIIZZIIIII', 'IIIIZZIIIIII', 'IIIZZIIIIIII', 'IIZZIIIIIIII', 'IZZIIIIIIIII', 'ZZIIIIIIIIII', 'IIIIIIIIIIIX', 'IIIIIIIIIIXI', 'IIIIIIIIIXII', 'IIIIIIIIXIII', 'IIIIIIIXIIII', 'IIIIIIXIIIII', 'IIIIIXIIIIII', 'IIIIXIIIIIII', 'IIIXIIIIIIII', 'IIXIIIIIIIII', 'IXIIIIIIIIII', 'XIIIIIIIIIII'],
              coeffs=[ 1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,
  1.+0.j,  1.+0.j,  1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j,
 -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j])


If we would like to measure the energy the observable is the Hamiltonian itself. Alternatively, we could be
interested in measuring system properties like the average magnetization by counting the number of spins
aligned in the $Z$-direction with the observable

$$
O = \frac{1}{n} \sum_{i=1} Z_i
$$

For observables that are not given in terms of Pauli operators but in a matrix form, we first have to reformulate them
in the Pauli basis in order to evaluate them on a quantum computer.
We are always able to find such a representation as the Pauli matrices form a basis for the Hermitian $2^n \times 2^n$ matrices.
We expand the observable $O$ as

$$
O = \sum_{P \in \{I, X, Y, Z\}^{\otimes n}} \mathrm{Tr}(O P) P,
$$

where the sum runs over all possible $n$-qubit Pauli terms and $\mathrm{Tr}(\cdot)$ is the trace of a matrix, which acts as inner product.
You can implement this decomposition  from a matrix to Pauli terms using the `SparsePauliOp.from_operator` method, like so:

In [2]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp

matrix = np.array(
    [[-1, 0, 0.5, -1], [0, 1, 1, 0.5], [0.5, 1, -1, 0], [-1, 0.5, 0, 1]]
)

observable = SparsePauliOp.from_operator(matrix)
print(observable)

SparsePauliOp(['IZ', 'XI', 'YY'],
              coeffs=[-1. +0.j,  0.5+0.j,  1. -0.j])


Jeśli chcemy zmierzyć energię, obserwablą jest sam hamiltoniana. Alternatywnie możemy być zainteresowani pomiarem właściwości systemu, takich jak średnie namagnesowanie, poprzez liczenie spinów wyrównanych w kierunku $Z$ za pomocą obserwabli

$$
O = \frac{1}{n} \sum_{i=1} Z_i
$$

Dla obserwabli, które nie są podane w kategoriach operatorów Pauliego, lecz w postaci macierzowej, musimy najpierw sformułować je w bazie Pauliego, aby ocenić je na komputerze kwantowym.
Zawsze jesteśmy w stanie znaleźć taką reprezentację, ponieważ macierze Pauliego tworzą bazę dla hermitowskich macierzy $2^n \times 2^n$.
Rozwijamy obserwablę $O$ jako

$$
O = \sum_{P \in {I, X, Y, Z}^{\otimes n}} \mathrm{Tr}(O P) P,
$$

gdzie suma przebiega po wszystkich możliwych $n$-qubitowych składowych Pauliego, a $\mathrm{Tr}(\cdot)$ jest śladem macierzy, który pełni rolę iloczynu skalarnego.
Tę dekompozycję z macierzy na składowe Pauliego możesz zaimplementować za pomocą metody `SparsePauliOp.from_operator`, jak poniżej:

In [3]:
from qiskit.circuit import QuantumCircuit

# create a circuit, where we would like to measure
# q0 in the X basis, q1 in the Y basis and q2 in the Z basis
circuit = QuantumCircuit(3)
circuit.ry(0.8, 0)
circuit.cx(0, 1)
circuit.cx(1, 2)
circuit.barrier()

# diagonalize X with the Hadamard gate
circuit.h(0)

# diagonalize Y with Hadamard as S^\dagger
circuit.sdg(1)
circuit.h(1)

# the Z basis is the default, no action required here

# measure all qubits
circuit.measure_all()
circuit.draw("mpl")

<Image src="../docs/images/guides/specify-observables-pauli/extracted-outputs/ce4b1984-ebe0-44f6-a78c-d67b9e9bb361-0.svg" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
  -  Read the [SparsePauliOp API](/docs/api/qiskit/qiskit.quantum_info.SparsePauliOp#sparsepauliop) reference.
</Admonition>